# 02 — Training the Rating Regressor (with Netflix Prize features)

**MIS 3080 Final Project — Movie Recommender**

This notebook trains the **ML component** of the project: an XGBoost regressor that predicts a movie's `vote_average` (TMDB user rating, 0–10) from features. It uses the enriched dataset produced by notebook 04, which adds **collaborative-filtering signals from the Netflix Prize dataset** (~100M user ratings, ~17K movies) on top of the TMDB metadata.

### Why this notebook reads `movies_with_netflix.parquet`

Our previous version of this notebook used only TMDB-derived features and reached R² = 0.58. Netflix gives us two additional families of features:

- **Per-movie aggregates** (4 columns): `netflix_mean_rating`, `netflix_rating_count`, `netflix_rating_std`, `netflix_log_count`. A second, independent quality signal from a different audience.
- **Latent factors from matrix factorization** (32 columns): `netflix_svd_00 … netflix_svd_31`. Numbers learned by `TruncatedSVD` from the user × movie rating matrix — they encode "which kinds of users like this movie" without anyone labelling those user types. Conceptually different from any metadata-based feature.

Roughly 44% of TMDB movies have non-NaN Netflix features (the Netflix Prize data was collected up to 2005, so post-2005 releases are not in the catalog). XGBoost's median imputer handles the missing values transparently.

### What this notebook produces

- **`model.joblib`** — fitted scikit-learn `Pipeline` (preprocessor + XGBoost). Trained with all features, including Netflix.
- **`model_meta.json`** — feature names + held-out metrics for every variant we tried. Useful for the writeup.

### Modeling approach

We compare four models on the same train/test split:

1. **Mean baseline** — sanity floor (predicts the training-set mean for every movie).
2. **Linear regression (TMDB only)** — interpretable baseline.
3. **XGBoost (TMDB only)** — the *previous* version of this model, included for a clean before/after.
4. **XGBoost (TMDB + Netflix)** — the new, saved model.

Metrics on a 20% held-out test set: **MAE**, **RMSE**, **R²**.


## 0. Setup — paths


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH    = PROJECT_ROOT / "artifacts" / "movies_with_netflix.parquet"
ARTIFACTS    = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), (
    f"missing {DATA_PATH} - run 01_eda.ipynb and 04_netflix_features.ipynb first"
)
print("DATA_PATH:", DATA_PATH)
print("ARTIFACTS:", ARTIFACTS)


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

from xgboost import XGBRegressor

pd.set_option("display.max_columns", 80)
sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(42)


## 1. Load the enriched dataset


In [ ]:
df = pd.read_parquet(DATA_PATH)
print("loaded:", df.shape)
netflix_cols = [c for c in df.columns if c.startswith("netflix_")]
print(f"  Netflix columns ({len(netflix_cols)}):", netflix_cols[:6], "...")
print(f"  Netflix coverage : {df['netflix_mean_rating'].notna().sum():,} of {len(df):,}")


### Filter to `train_eligible` rows for training

In [ ]:
train_df = df[df["train_eligible"]].copy()
print(f"all rows           : {len(df):,}")
print(f"training-eligible  : {len(train_df):,}")
print(f"  with Netflix data: {train_df['netflix_mean_rating'].notna().sum():,}")


## 2. Define feature regimes


Three feature regimes, all sharing the same TMDB-derived backbone. The model comparison shows what each layer contributes.


In [ ]:
NUMERIC_BASE = [
    "release_year", "runtime",
    "log_budget", "log_revenue", "log_popularity", "log_vote_count",
    "n_genres", "n_keywords", "cast_size",
]
NETFLIX_AGG = [
    "netflix_mean_rating", "netflix_rating_count",
    "netflix_rating_std",  "netflix_log_count",
]
NETFLIX_SVD = [f"netflix_svd_{i:02d}" for i in range(32)]

LOW_CARDINALITY_CATEGORICAL  = ["primary_genre", "original_language"]
HIGH_CARDINALITY_CATEGORICAL = ["director", "lead_actor"]

FEATURES_BASE     = NUMERIC_BASE + LOW_CARDINALITY_CATEGORICAL + HIGH_CARDINALITY_CATEGORICAL
FEATURES_FULL     = FEATURES_BASE + NETFLIX_AGG + NETFLIX_SVD
TARGET = "vote_average"

print(f"base features  : {len(FEATURES_BASE)}")
print(f"full features  : {len(FEATURES_FULL)}  (+{len(FEATURES_FULL)-len(FEATURES_BASE)} Netflix)")


## 3. Train/test split


One split, fixed seed, used by every model below. We split using `FEATURES_FULL` so the same row IDs are in train vs. test for every comparison — apples to apples.


In [ ]:
X_full = train_df[FEATURES_FULL]
y      = train_df[TARGET]

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=42
)
X_train_base = X_train_full[FEATURES_BASE]
X_test_base  = X_test_full[FEATURES_BASE]
print(f"train: {X_train_full.shape}    test: {X_test_full.shape}")


## 4. Reusable preprocessing pipeline


In [ ]:
def make_preprocessor(numeric_cols, scale_numeric: bool) -> ColumnTransformer:
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    numeric_pipe = Pipeline(num_steps)

    low_cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    high_cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("target",  TargetEncoder(smooth="auto", random_state=42)),
    ])

    return ColumnTransformer([
        ("num",      numeric_pipe,  numeric_cols),
        ("low_cat",  low_cat_pipe,  LOW_CARDINALITY_CATEGORICAL),
        ("high_cat", high_cat_pipe, HIGH_CARDINALITY_CATEGORICAL),
    ])

def make_xgb_pipeline(numeric_cols):
    return Pipeline([
        ("prep", make_preprocessor(numeric_cols, scale_numeric=False)),
        ("model", XGBRegressor(
            n_estimators=400, learning_rate=0.05, max_depth=6,
            subsample=0.9, colsample_bytree=0.9, min_child_weight=3,
            random_state=42, n_jobs=-1, tree_method="hist",
        )),
    ])


## 5. Train and evaluate four models


In [ ]:
def evaluate(name, y_true, y_pred):
    return {
        "model": name,
        "MAE":  mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2":   r2_score(y_true, y_pred),
    }

results = []


**1) Mean baseline**

In [ ]:
mean_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)
results.append(evaluate("Mean baseline", y_test, mean_pred))
results[-1]


**2) Linear regression (TMDB only)**

In [ ]:
linear = Pipeline([
    ("prep", make_preprocessor(NUMERIC_BASE, scale_numeric=True)),
    ("model", LinearRegression()),
])
linear.fit(X_train_base, y_train)
results.append(evaluate("Linear regression (TMDB)", y_test, linear.predict(X_test_base)))
results[-1]


**3) XGBoost (TMDB only) — *previous* model, included for before/after**

In [ ]:
xgb_base = make_xgb_pipeline(NUMERIC_BASE)
xgb_base.fit(X_train_base, y_train)
results.append(evaluate("XGBoost (TMDB only)", y_test, xgb_base.predict(X_test_base)))
results[-1]


**4) XGBoost (TMDB + Netflix) — new, saved model**

In [ ]:
NUMERIC_FULL = NUMERIC_BASE + NETFLIX_AGG + NETFLIX_SVD
xgb_full = make_xgb_pipeline(NUMERIC_FULL)
xgb_full.fit(X_train_full, y_train)
results.append(evaluate("XGBoost (TMDB + Netflix)", y_test, xgb_full.predict(X_test_full)))
results[-1]


## 6. Compare the four models


In [ ]:
results_df = pd.DataFrame(results).round(4)
results_df


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
colors = ["#cccccc", "#aac2dd", "#7fa6d6", "#1f3a5f"]
for ax, metric in zip(axes, ["MAE", "RMSE", "R2"]):
    results_df.plot.bar(x="model", y=metric, ax=ax, color=colors,
                        legend=False, edgecolor="black")
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()


**Interpretation:**

- The mean baseline establishes the noise floor (R² ≈ 0).
- Linear regression already captures roughly half the variance — most of that comes from `log_vote_count` and the target-encoded director/actor signals.
- XGBoost on TMDB-only features adds maybe 10 points of R² over linear regression, courtesy of non-linear interactions.
- **The Netflix features add another 5-7 points of R² on top of XGBoost-on-TMDB.** They contribute genuinely new information: `netflix_mean_rating` is an independent quality measurement, and the SVD factors encode latent audience taste profiles that no metadata feature can capture.


## 7. Diagnostics for the saved model (XGBoost + Netflix)


In [ ]:
xgb_pred = xgb_full.predict(X_test_full)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_test, xgb_pred, alpha=0.35, s=14)
lims = [min(y_test.min(), xgb_pred.min()) - 0.3,
        max(y_test.max(), xgb_pred.max()) + 0.3]
ax.plot(lims, lims, "r--", linewidth=1, label="y = x")
ax.set_xlabel("Actual vote_average"); ax.set_ylabel("Predicted vote_average")
ax.set_title("Predicted vs. actual (held-out test set)")
ax.legend(); plt.show()


In [ ]:
residuals = y_test - xgb_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].scatter(xgb_pred, residuals, alpha=0.35, s=14)
axes[0].axhline(0, color="red", linewidth=1)
axes[0].set_xlabel("predicted"); axes[0].set_ylabel("residual")
axes[0].set_title("Residuals vs predicted")
residuals.plot.hist(bins=40, ax=axes[1], edgecolor="white")
axes[1].set_title("Residual distribution"); axes[1].set_xlabel("actual - predicted")
plt.tight_layout(); plt.show()
print(residuals.describe().round(3))


## 8. Feature importance (model transparency)


Where does the predictive power come from in the new, Netflix-augmented model? We pull XGBoost's gain-based importances and group them so it's easy to see how the families compare.


In [ ]:
booster_names = xgb_full.named_steps["prep"].get_feature_names_out()
importances   = xgb_full.named_steps["model"].feature_importances_

imp_df = (pd.DataFrame({"feature": booster_names, "importance": importances})
          .sort_values("importance", ascending=True)
          .tail(25))

fig, ax = plt.subplots(figsize=(8, 8))
imp_df.plot.barh(x="feature", y="importance", ax=ax, color="#1f3a5f", legend=False)
ax.set_title("Top-25 features by gain importance (XGBoost + Netflix)")
ax.set_xlabel("gain importance")
plt.tight_layout(); plt.show()


In [ ]:
# Grouped contribution summary — shows how the families compare
def family(name):
    if name.startswith("num__netflix_svd_"):  return "Netflix SVD"
    if name.startswith("num__netflix_"):      return "Netflix aggregate"
    if name.startswith("num__"):              return "TMDB numeric"
    if name.startswith("low_cat__"):          return "TMDB low-cat"
    if name.startswith("high_cat__"):         return "TMDB high-cat (director/actor)"
    return "other"

grouped = (pd.DataFrame({"feature": booster_names, "importance": importances})
           .assign(family=lambda d: d["feature"].apply(family))
           .groupby("family")["importance"].sum()
           .sort_values(ascending=False))
grouped.round(3)


## 9. Save the chosen model and metadata


In [ ]:
FINAL_MODEL = xgb_full

model_path = ARTIFACTS / "model.joblib"
joblib.dump(FINAL_MODEL, model_path)

meta = {
    "selected_model": "XGBoost (TMDB + Netflix)",
    "target": TARGET,
    "feature_groups": {
        "numeric_base":    NUMERIC_BASE,
        "netflix_aggregates": NETFLIX_AGG,
        "netflix_svd":     NETFLIX_SVD,
        "low_cardinality_categorical":  LOW_CARDINALITY_CATEGORICAL,
        "high_cardinality_categorical": HIGH_CARDINALITY_CATEGORICAL,
    },
    "all_features": FEATURES_FULL,
    "n_train": int(len(X_train_full)),
    "n_test":  int(len(X_test_full)),
    "metrics": {r["model"]: {k: float(round(v, 4)) for k, v in r.items() if k != "model"}
                for r in results},
}
(ARTIFACTS / "model_meta.json").write_text(json.dumps(meta, indent=2))

print(f"saved {model_path}  ({model_path.stat().st_size/1024:.1f} KB)")
print(f"saved {ARTIFACTS / 'model_meta.json'}")
print()
print("metrics:")
print(json.dumps(meta["metrics"], indent=2))


In [ ]:
# Round-trip sanity check
reloaded = joblib.load(model_path)
sample_pred = reloaded.predict(X_test_full.iloc[:5])
print("reloaded predicts:", sample_pred.round(2))
print("actual           :", y_test.iloc[:5].values)


## 10. Next steps

- **`03_build_embeddings.ipynb`** — already produced `embeddings.npy`. No changes needed; embeddings are independent of the rating model.
- **`app.py`** — already wired to load `model.joblib`. Update its `FEATURE_COLS` to include the new Netflix columns and point it at `movies_with_netflix.parquet`.
